# AI Disclosure in UK Annual Reports: Pilot Study

**How to run:** click **Runtime > Run all** (on a phone: the menu, then Runtime, then Run all). It takes about 10 to 20 minutes.

When it finishes, a file called **pilot_results.zip** downloads (or appears in the Files panel on the left). Send that small file back in the chat. (Version 2: fixes broken 'fi' letters in some PDFs and splits bullet lists.)

What it does:
1. Downloads each annual report directly from the company or archive website.
2. Converts each PDF to text and counts words.
3. Finds every sentence that mentions AI.
4. Labels each AI sentence as specific or generic.
5. Summarises by year and draws the chart for the proposal.

In [ ]:
# STEP 0: install the PDF reader (pandas and matplotlib are already in Colab)
!pip -q install pypdf

In [ ]:
# STEP 1: settings

AI_TERMS = [
    "artificial intelligence", "machine learning", "deep learning", "neural network",
    "generative ai", "gen ai", "genai", "large language model", "natural language processing",
    "computer vision", "predictive analytics", "chatgpt", "copilot", "chatbot",
]
MATCH_STANDALONE_AI = True   # also count the word "AI" on its own, in capitals

SPECIFIC_SIGNALS = [
    "£", "$", "€", "%", "million", "bn", "billion",
    "invested", "investment in", "launched", "deployed", "rolled out",
    "partnership with", "partnered with", "acquired", "acquisition of",
    "pilot", "reduced", "increased", "saved", "hours",
    "colleagues trained", "employees trained", "use cases",
]

SHOCK_YEAR = 2023   # first report year after ChatGPT's release (30 Nov 2022)

# Candidate download links for each report. The notebook tries them in order
# and keeps the first one that returns a real PDF. Some are expected to fail.
RELX = "https://www.relx.com/~/media/Files/R/RELX-Group/documents/reports/annual-reports/relx-{y}-annual-report.pdf"
SAGE = "https://www.sage.com/en-gb/-/media/Files/investors/documents/pdf/annual%20report/sage-annual-report-{y}.pdf"
ULVR = "https://www.unilever.com/files/unilever-annual-report-and-accounts-{y}.pdf"
RR = "https://www.rolls-royce.com/~/media/Files/R/Rolls-Royce/documents/annual-report/{y}/rr-plc-annual-report-{y}.pdf"
RR2 = "https://www.mylearning.rolls-royce.com/~/media/Files/R/Rolls-Royce/documents/annual-report/{y}/rr-plc-annual-report-{y}.pdf"
SHELL = "https://reports.shell.com/annual-report/{y}/_assets/downloads/shell-annual-report-{y}.pdf"
UAB = "https://ddd.uab.cat/pub/infanu/{folder}/ia{code}a{y}ieng.pdf"

CANDIDATES = {
    ("TSCO", 2019): ["https://www.tescoplc.com/media/476423/tesco_ar_2019.pdf"],
    ("TSCO", 2021): ["https://www.tescoplc.com/media/757589/tesco_annual_report_2021.pdf"],
    ("TSCO", 2023): ["https://www.tescoplc.com/ar2023"],
    ("TSCO", 2025): ["https://www.tescoplc.com/media/ky0bfwpo/tesco_ar25_interactive.pdf"],

    ("BT", 2019): [UAB.format(folder=119167, code="BT", y=2019)],
    ("BT", 2021): [UAB.format(folder=119167, code="BT", y=2021)],
    ("BT", 2023): ["https://www.bt.com/about/annual-reports/2023summary/assets/files/bt_ar23_annual_report_2023.pdf",
                   UAB.format(folder=119167, code="BT", y=2023)],
    ("BT", 2025): ["https://www.bt.com/bt-plc/assets/documents/investors/financial-reporting-and-news/annual-reports/2025/2025-bt-group-plc-annual-report.pdf",
                   UAB.format(folder=119167, code="BT", y=2025)],

    ("AZN", 2019): [UAB.format(folder=40172, code="ASTZEN", y=2019)],
    ("AZN", 2021): [UAB.format(folder=40172, code="ASTZEN", y=2021)],
    ("AZN", 2023): ["https://www.ecgi.global/system/files/2024-04/astrazeneca_ar_2023.pdf",
                    UAB.format(folder=40172, code="ASTZEN", y=2023)],
    ("AZN", 2025): [UAB.format(folder=40172, code="ASTZEN", y=2025)],

    ("DGE", 2019): [UAB.format(folder=43622, code="DIAGEO", y=2019)],
    ("DGE", 2021): ["https://media.diageo.com/diageo-corporate-media/media/tyhd4gms/273_diageo-annual-report-2021.pdf",
                    UAB.format(folder=43622, code="DIAGEO", y=2021)],
    ("DGE", 2023): ["https://staticcontents.investis.com/files/d/diageo/diageo-annual-report-2023.pdf",
                    UAB.format(folder=43622, code="DIAGEO", y=2023)],
    ("DGE", 2025): ["https://www.diageo.com/~/media/Files/D/Diageo-V2/Diageo-Corp/investors/results-reports-and-events/annual-reports/2025/annual-report-2025.pdf",
                    UAB.format(folder=43622, code="DIAGEO", y=2025)],
}
for y in (2019, 2021, 2023, 2025):
    CANDIDATES[("REL", y)] = [RELX.format(y=y)]
    CANDIDATES[("SGE", y)] = [SAGE.format(y=y)]
    CANDIDATES[("ULVR", y)] = [ULVR.format(y=y)]
    CANDIDATES[("RR", y)] = [RR.format(y=y), RR2.format(y=y)]
    CANDIDATES[("SHEL", y)] = [SHELL.format(y=y)]

print(len(CANDIDATES), "reports to try")

In [ ]:
# STEP 2: download the reports

import time
from pathlib import Path
import requests
import pandas as pd

REPORTS_DIR = Path("reports"); REPORTS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path("output"); OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/pdf,*/*",
}

def download(urls, dest):
    """Try each URL; keep the first response that is really a PDF."""
    for url in urls:
        try:
            r = requests.get(url, headers=HEADERS, timeout=90, allow_redirects=True)
            if r.status_code == 200 and r.content[:5] == b"%PDF-":
                dest.write_bytes(r.content)
                return url, "ok", len(r.content)
            note = f"HTTP {r.status_code}" if r.status_code != 200 else "not a PDF"
        except Exception as error:
            note = type(error).__name__
        time.sleep(1)
    return urls[-1], note, 0

log = []
for (ticker, year), urls in sorted(CANDIDATES.items()):
    dest = REPORTS_DIR / f"{ticker}_{year}.pdf"
    if dest.exists():
        log.append({"ticker": ticker, "year": year, "status": "ok", "mb": round(dest.stat().st_size / 1e6, 1), "source": "already downloaded"})
        continue
    url, status, size = download(urls, dest)
    log.append({"ticker": ticker, "year": year, "status": status, "mb": round(size / 1e6, 1), "source": url})
    print(f"{ticker} {year}: {status}")

download_log = pd.DataFrame(log)
download_log.to_csv(OUTPUT_DIR / "download_log.csv", index=False)
print("\nDownloaded:", (download_log["status"] == "ok").sum(), "of", len(download_log))

In [ ]:
# STEP 3: PDF to text, word counts

import re
import unicodedata
from pypdf import PdfReader

def clean(text):
    """Fix PDF ligatures so words like 'artificial' are not broken up.
    Some PDFs store 'fi' as a single symbol, which comes out as '/f_i' or 'ﬁ'."""
    text = unicodedata.normalize("NFKC", text)          # turns 'ﬁ' into 'fi'
    for broken, fixed in (("/f_f_i", "ffi"), ("/f_f_l", "ffl"), ("/f_f", "ff"), ("/f_i", "fi"), ("/f_l", "fl")):
        text = text.replace(broken, fixed)
    return text

texts, words = {}, []
for pdf in sorted(REPORTS_DIR.glob("*.pdf")):
    ticker, year = pdf.stem.rsplit("_", 1)
    try:
        reader = PdfReader(pdf)
        text = "\n".join((p.extract_text() or "") for p in reader.pages)
        text = clean(text)
    except Exception as error:
        print("Could not read", pdf.name, error); continue
    n_words = len(re.findall(r"[A-Za-z]+", text))
    texts[(ticker, int(year))] = text
    words.append({"ticker": ticker, "year": int(year), "pages": len(reader.pages), "words": n_words})
    flag = "  <- very little text, check" if n_words < 5000 else ""
    print(f"{ticker} {year}: {len(reader.pages)} pages, {n_words:,} words{flag}")

word_counts = pd.DataFrame(words)
word_counts.to_csv(OUTPUT_DIR / "word_counts.csv", index=False)

In [ ]:
# STEP 4: find AI sentences and label them specific or generic

STANDALONE_AI = re.compile(r"\bAI\b")

def split_sentences(text):
    text = re.sub(r"\s+", " ", text)
    # split at sentence ends, and also at bullet points, so lists are not merged into one long "sentence"
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z])|\s[•▪●]\s|\s>\s(?=[A-Z])", text)
    return [p.strip() for p in parts if 30 <= len(p) <= 1000]

def matched_terms(sentence):
    lower = sentence.lower()
    found = [t for t in AI_TERMS if t in lower]
    if MATCH_STANDALONE_AI and STANDALONE_AI.search(sentence):
        found.append("AI")
    return found

def is_specific(sentence):
    lower = sentence.lower()
    return any(ch.isdigit() for ch in sentence) or any(s in lower for s in SPECIFIC_SIGNALS)

rows = []
for (ticker, year), text in texts.items():
    for s in split_sentences(text):
        terms = matched_terms(s)
        if terms:
            rows.append({"ticker": ticker, "year": year, "terms": "; ".join(terms),
                         "label": "specific" if is_specific(s) else "generic", "sentence": s})

ai_sentences = pd.DataFrame(rows, columns=["ticker", "year", "terms", "label", "sentence"])
ai_sentences.to_csv(OUTPUT_DIR / "ai_sentences.csv", index=False)
print(len(ai_sentences), "AI sentences found")
print(ai_sentences["label"].value_counts())

In [ ]:
# STEP 5: summaries and chart

import matplotlib.pyplot as plt

counts = (ai_sentences.assign(specific=ai_sentences["label"].eq("specific"))
          .groupby(["ticker", "year"])
          .agg(ai_sentences=("sentence", "size"), specific_sentences=("specific", "sum"))
          .reset_index())
by_report = word_counts.merge(counts, on=["ticker", "year"], how="left").fillna({"ai_sentences": 0, "specific_sentences": 0})
by_report["ai_per_10k_words"] = by_report["ai_sentences"] / by_report["words"] * 10_000
by_report["mentions_ai"] = by_report["ai_sentences"] > 0
by_report.to_csv(OUTPUT_DIR / "by_report.csv", index=False)

by_year = (by_report.groupby("year")
           .agg(reports=("ticker", "size"),
                pct_reports_mentioning_ai=("mentions_ai", "mean"),
                mean_ai_per_10k_words=("ai_per_10k_words", "mean"),
                median_ai_per_10k_words=("ai_per_10k_words", "median"),
                total_ai_sentences=("ai_sentences", "sum"),
                total_specific=("specific_sentences", "sum"))
           .reset_index())
by_year["pct_reports_mentioning_ai"] *= 100
by_year["pct_specific"] = by_year["total_specific"] / by_year["total_ai_sentences"].where(by_year["total_ai_sentences"] > 0) * 100
by_year.to_csv(OUTPUT_DIR / "by_year.csv", index=False)

# Balanced panel check: companies with all four years downloaded
complete = by_report.groupby("ticker")["year"].nunique()
balanced = by_report[by_report["ticker"].isin(complete[complete == 4].index)]
balanced_by_year = balanced.groupby("year")["ai_per_10k_words"].agg(["count", "mean", "median"]).reset_index()
balanced_by_year.to_csv(OUTPUT_DIR / "by_year_balanced.csv", index=False)

display(by_year.round(2))
display(balanced_by_year.round(2))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(by_year["year"], by_year["mean_ai_per_10k_words"], marker="o", color="#1F3864", label="All reports")
if len(balanced):
    ax.plot(balanced_by_year["year"], balanced_by_year["mean"], marker="s", linestyle=":", color="#7A8FB8",
            label=f"Same {balanced['ticker'].nunique()} companies every year")
ax.axvline(SHOCK_YEAR - 0.5, linestyle="--", color="grey")
ax.text(SHOCK_YEAR - 0.45, ax.get_ylim()[1] * 0.9, "ChatGPT released\n(Nov 2022)", fontsize=9, color="grey")
ax.set_xlabel("Annual report year"); ax.set_ylabel("AI sentences per 10,000 words (mean)")
ax.set_title(f"AI disclosure in FTSE 100 annual reports ({len(by_report)} reports)")
ax.set_xticks(by_year["year"]); ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "ai_disclosure_chart.png", dpi=200); plt.show()

In [ ]:
# STEP 6: package the results (small file) and download it

import shutil
shutil.make_archive("pilot_results", "zip", OUTPUT_DIR)
try:
    from google.colab import files
    files.download("pilot_results.zip")
except Exception:
    print("Download pilot_results.zip from the Files panel on the left.")